In [2]:
from pathlib import Path
import pandas as pd
import joblib
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models" / "traditional_ml"

test_df = pd.read_csv(DATA_DIR / "test.csv")
required_columns = {"clean_text", "sentiment"}
missing_columns = required_columns - set(test_df.columns)
if missing_columns:
    raise ValueError(f"Test data is missing columns: {sorted(missing_columns)}")

test_df["clean_text"] = test_df["clean_text"].fillna("").astype(str)
test_df["sentiment"] = test_df["sentiment"].astype(str).str.strip().str.lower()

vectorizer = joblib.load(MODEL_DIR / "tfidf_vectorizer.joblib")
model = joblib.load(MODEL_DIR / "best_tfidf_model.joblib")
X_test = vectorizer.transform(test_df["clean_text"])
test_df["predicted_sentiment"] = (
    pd.Series(model.predict(X_test), index=test_df.index)
    .astype(str).str.strip().str.lower()
)

errors = test_df[
    test_df["sentiment"] != test_df["predicted_sentiment"]
].copy()

print("Total test samples:          ", len(test_df))
print("Total incorrect predictions: ", len(errors))
print(f"Overall Error Rate:           {len(errors) / len(test_df):.2%}")

print("\n--- Error Breakdown by True vs Predicted Sentiment ---")
error_matrix = pd.crosstab(
    errors["sentiment"],
    errors["predicted_sentiment"],
    margins=True
)
display(error_matrix)

if "rating" in errors.columns:
    print("\n--- Error Distribution Across Star Ratings ---")
    display(errors["rating"].value_counts().sort_index())

print("\n--- Sample Misclassified Reviews ---")
review_column = "text" if "text" in errors.columns else "clean_text"
display_columns = [review_column, "sentiment", "predicted_sentiment"]
if "rating" in errors.columns:
    display_columns.append("rating")
display(errors[display_columns].head(20))


d:\My_Project\Analyse\mobile-review-sentiment-analysis\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\My_Project\Analyse\mobile-review-sentiment-analysis\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\My_Project\Analyse\mobile-review-sentiment-analysis\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Tryin

Total test samples:           1999
Total incorrect predictions:  412
Overall Error Rate:           20.61%

--- Error Breakdown by True vs Predicted Sentiment ---


predicted_sentiment,negative,neutral,positive,All
sentiment,,,,
negative,0,68,50,118
neutral,48,0,92,140
positive,53,101,0,154
All,101,169,142,412



--- Sample Misclassified Reviews ---


,text,sentiment,predicted_sentiment
10,I primarily bought this headset for using with...,neutral,positive
14,The product arrived on time and is as describe...,positive,neutral
17,"Can't really complain too much, because c'mon ...",neutral,negative
18,great for mounting on windshields. It comes w...,positive,negative
23,"I ordered this item and the clear, hard case a...",neutral,positive
31,Mophie should be ashamed. This case does not ...,negative,positive
34,The TENQA HP-109 are nice stereo bluetooth hea...,positive,neutral
35,"I've boughten two of these now, they work grea...",positive,neutral
38,My ears are to small for this bluetooth. I ret...,positive,negative
40,"When I purchased this set, the description sta...",neutral,negative
